# 5. Unresolved after evidence review

This saved review has structured judgments, but some targets remain ineligible because their required evidence has unresolved dependencies. We replay the final saved eligibility decision without asking a model to review it again.

Run cells from top to bottom. Everything runs offline; no API key or model request is used.

In [ ]:
from pathlib import Path
import sys
import json
from pprint import pprint

# Works when opened from the repository, notebooks/, or its parent directory.
root = next(p for base in [Path.cwd(), *Path.cwd().parents]
            for p in [base, base / "research_finder"]
            if (p / "novelty/comparison_workflow.py").exists())
sys.path.insert(0, str(root))
# Check the selected kernel before importing project modules.
import importlib.util
missing = [name for name in ("openai", "pydantic", "pymupdf")
           if importlib.util.find_spec(name) is None]
if missing:
    raise RuntimeError(
        f"Kernel {sys.executable} is missing: {', '.join(missing)}. "
        f"In VS Code choose Select Kernel > Python Environments > {root / '.venv/bin/python'}, "
        "then restart the kernel and Run All."
    )
from notebooks.example_helpers import load_example, expect_failure
from extraction.research_extract import _parse_json
from novelty.comparison_records import EvidenceReport, EvidenceRecord, EvidenceReview
from novelty.comparison_schemas import Passage
from novelty.comparison_workflow import validate_evidence_report, validate_record, eligible, scope_citation_gaps
from novelty.schemas import TargetSignature


In [ ]:
example = load_example(root, "05_unresolved_evidence_review")
pprint(example["provenance"])

In [ ]:
review = EvidenceReview.model_validate(example["review"])
print("Saved diagnostic:", example["recorded_error"])
print("Review summary:", review.report.summary)
allowed, supported_claims = eligible(review)
gaps = scope_citation_gaps(review.report, review.record)
print("Eligible targets:", sorted(allowed))
checks = {c.claim_id: c for c in review.report.claim_checks}
coverage = {c.target_id: c for c in review.report.coverage_checks}
for relationship in review.record.relationships:
    pprint({"target": relationship.target_id, "coverage_review": coverage[relationship.target_id].decision, "eligible": relationship.target_id in allowed, "scope_citation_gaps": {cid: gaps[cid] for cid in sorted(relationship.refs()) if cid in gaps}, "unsupported_claims": {cid: checks[cid].decision for cid in sorted(relationship.refs()) if checks[cid].decision != "supported"}})
assert len(allowed) < len(review.record.relationships)

## Inspect one actual blocking claim and its passages
An `adequate` coverage judgment is only one requirement. Every claim used by that target must also be supported and have its scope evidence attached.

In [ ]:
blocked = next(r for r in review.record.relationships if r.target_id not in allowed)
cid = next((cid for cid in sorted(blocked.refs()) if cid in gaps or checks[cid].decision != "supported"), None)
if cid is not None:
    claim = next(c for c in review.record.claims if c.claim_id == cid)
    pprint(claim.model_dump())
    pprint(checks[cid].model_dump())
    ids = set(claim.passage_ids) | set(gaps.get(cid, []))
    for passage in example["passages"]:
        if passage["passage_id"] in ids:
            pprint(passage)
else:
    pprint(coverage[blocked.target_id].model_dump())

Unresolved means this comparison cannot yet supply an accepted judgment. It does not mean the hypothesis was disproved or already studied. This notebook replays recorded model judgments and deterministic gates; it does not independently establish whether those judgments are correct.